In [1]:
import os
from dotenv import load_dotenv
from langchain_community.document_loaders import DirectoryLoader, TextLoader

load_dotenv()
print("Anahtar yüklendi:", bool(os.getenv("GEMINI_API_KEY")))

loader = DirectoryLoader("data", glob="*.txt", loader_cls=TextLoader, loader_kwargs={"encoding": "utf-8"})
documents = loader.load()

print(len(documents), "belge yüklendi")
for doc in documents:
    print(doc.metadata["source"], "->", len(doc.page_content), "karakter")

Anahtar yüklendi: True
3 belge yüklendi
data/ekipman_politikasi.txt -> 340 karakter
data/izin_politikasi.txt -> 343 karakter
data/uzaktan_calisma.txt -> 399 karakter


In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=40)
chunks = splitter.split_documents(documents)

print(len(documents), "belge ->", len(chunks), "parça")
for i, chunk in enumerate(chunks):
    print(f"\n[{i}] {chunk.metadata['source']} ({len(chunk.page_content)} karakter)")
    print(chunk.page_content)

3 belge -> 9 parça

[0] data/ekipman_politikasi.txt (37 karakter)
Mavi Kod Yazılım - Ekipman Politikası

[1] data/ekipman_politikasi.txt (179 karakter)
Yeni çalışanlara dizüstü bilgisayar verilir. Bilgisayarlar 4 yılda bir yenilenir.
Monitör ve ergonomik sandalye için çalışan başına tek seferlik 12.000 TL ekipman bütçesi ayrılır.

[2] data/ekipman_politikasi.txt (120 karakter)
Zimmetli ekipman, işten ayrılırken 5 iş günü içinde iade edilir.
Kaybolan ekipman 24 saat içinde BT birimine bildirilir.

[3] data/izin_politikasi.txt (41 karakter)
Mavi Kod Yazılım - Yıllık İzin Politikası

[4] data/izin_politikasi.txt (172 karakter)
Çalışanlar ilk 5 yıl boyunca yılda 14 iş günü, 5. yıldan sonra yılda 20 iş günü ücretli yıllık izin kullanır.
İzin talepleri en az 10 iş günü önceden yöneticiye bildirilir.

[5] data/izin_politikasi.txt (126 karakter)
Kullanılmayan izinlerin en fazla 5 günü ertesi yıla devredilebilir.
Her çalışan doğum gününde 1 gün ücretli izin kullanabilir.

[6] data/uzaktan_calis

In [3]:
from google import genai

client = genai.Client()

for model in client.models.list():
    if "embedContent" in (model.supported_actions or []):
        print(model.name)

models/gemini-embedding-001
models/gemini-embedding-2-preview
models/gemini-embedding-2


In [4]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

for name in ["models/gemini-embedding-2", "models/gemini-embedding-001"]:
    try:
        embeddings = GoogleGenerativeAIEmbeddings(model=name)
        vector = embeddings.embed_query("yıllık izin kaç gün")
        print(name, "-> çalışıyor, vektör boyutu:", len(vector))
    except Exception as e:
        print(name, "-> hata:", str(e)[:200])
        

models/gemini-embedding-2 -> çalışıyor, vektör boyutu: 3072
models/gemini-embedding-001 -> çalışıyor, vektör boyutu: 3072


In [5]:
from langchain_chroma import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

vectorstore = Chroma(persist_directory="chroma_db", embedding_function=embeddings)
vectorstore.delete_collection()

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="chroma_db",
)

print(vectorstore._collection.count(), "parça indekslendi")

9 parça indekslendi


In [6]:
questions = [
    "Yıllık izin kaç gün?",
    "Ofise hangi günler gelmem gerekiyor?",
    "Monitör için ne kadar bütçe var?",
]

for q in questions:
    print("\nSoru:", q)
    for doc, score in vectorstore.similarity_search_with_score(q, k=2):
        print(f"  mesafe={score:.3f} | {doc.metadata['source']} | {doc.page_content[:70]}...")


Soru: Yıllık izin kaç gün?
  mesafe=0.447 | data/izin_politikasi.txt | Çalışanlar ilk 5 yıl boyunca yılda 14 iş günü, 5. yıldan sonra yılda 2...
  mesafe=0.481 | data/izin_politikasi.txt | Kullanılmayan izinlerin en fazla 5 günü ertesi yıla devredilebilir.
He...

Soru: Ofise hangi günler gelmem gerekiyor?
  mesafe=0.433 | data/uzaktan_calisma.txt | Pazartesi ve cuma günleri uzaktan çalışılabilir. Salı, çarşamba ve per...
  mesafe=0.502 | data/uzaktan_calisma.txt | Ekip toplantıları saat 10:00'da başlar. Uzaktan katılımda kamerayı açm...

Soru: Monitör için ne kadar bütçe var?
  mesafe=0.573 | data/ekipman_politikasi.txt | Yeni çalışanlara dizüstü bilgisayar verilir. Bilgisayarlar 4 yılda bir...
  mesafe=0.728 | data/uzaktan_calisma.txt | Ekip toplantıları saat 10:00'da başlar. Uzaktan katılımda kamerayı açm...


In [7]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite", temperature=0)

prompt = ChatPromptTemplate.from_messages([
    ("system", "Sen Mavi Kod Yazılım'ın iç yönergelerine göre cevap veren bir asistansın. Sadece aşağıdaki bağlamı kullan. Cevap bağlamda yoksa 'Bu bilgi belgelerde yok.' de.\n\nBağlam:\n{context}"),
    ("human", "{question}"),
])

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

questions = [
    "5. yıldan sonra yılda kaç gün izin kullanabilirim?",
    "Monitör ve sandalye için ne kadar bütçe ayrılıyor?",
    "Yurt dışından en fazla kaç gün çalışabilirim?",
    "Şirketin yemek kartı uygulaması var mı?",
]

for q in questions:
    print("Soru:", q)
    print("Cevap:", rag_chain.invoke(q))
    print()

Soru: 5. yıldan sonra yılda kaç gün izin kullanabilirim?
Cevap: 5. yıldan sonra yılda 20 iş günü ücretli yıllık izin kullanabilirsiniz.

Soru: Monitör ve sandalye için ne kadar bütçe ayrılıyor?
Cevap: Monitör ve ergonomik sandalye için çalışan başına tek seferlik 12.000 TL ekipman bütçesi ayrılır.

Soru: Yurt dışından en fazla kaç gün çalışabilirim?
Cevap: Yurt dışından çalışma yılda en fazla 15 gün yapılabilir ve yönetici onayı gerektirir.

Soru: Şirketin yemek kartı uygulaması var mı?
Cevap: Bu bilgi belgelerde yok.



In [8]:
def ask(question):
    docs = retriever.invoke(question)
    answer = (prompt | llm | StrOutputParser()).invoke(
        {"context": format_docs(docs), "question": question}
    )
    print("Soru:", question)
    print("Cevap:", answer)
    print("Kaynaklar:", sorted({d.metadata["source"] for d in docs}))
    print()

for q in [
    "Uzaktan çalışırken internet desteği ne kadar?",
    "Yıllık izin kaç gün?",
    "Maaşlar ayın kaçında yatıyor?",
    "Doğum günümde izinli sayılır mıyım?",
]:
    ask(q)

Soru: Uzaktan çalışırken internet desteği ne kadar?
Cevap: Şirket, uzaktan çalışanlara aylık 500 TL internet desteği öder.
Kaynaklar: ['data/ekipman_politikasi.txt', 'data/uzaktan_calisma.txt']

Soru: Yıllık izin kaç gün?
Cevap: Çalışanlar ilk 5 yıl boyunca yılda 14 iş günü, 5. yıldan sonra yılda 20 iş günü ücretli yıllık izin kullanır.
Kaynaklar: ['data/izin_politikasi.txt']

Soru: Maaşlar ayın kaçında yatıyor?
Cevap: Bu bilgi belgelerde yok.
Kaynaklar: ['data/izin_politikasi.txt', 'data/uzaktan_calisma.txt']

Soru: Doğum günümde izinli sayılır mıyım?
Cevap: Her çalışan doğum gününde 1 gün ücretli izin kullanabilir.
Kaynaklar: ['data/izin_politikasi.txt']



## Sonuç

- **Belgeler:** Gerçek veri kullanmamak için kurgusal bir şirketin (Mavi Kod Yazılım) üç kısa iç yönergesi yazıldı (izin, uzaktan çalışma, ekipman). `DirectoryLoader` ve `TextLoader` ile yüklendi. Belgeler uydurma olduğu için modelin cevaplardaki rakamları (20 gün, 12.000 TL, 15 gün) kendi bilgisinden değil, retriever'ın getirdiği parçalardan bilebildiği garanti.

- **Parçalama:** `RecursiveCharacterTextSplitter` (`chunk_size=200`, `chunk_overlap=40`) 3 belgeyi 9 parçaya böldü. Hiçbir cümle ortadan kesilmedi. Satırlar 40 karakterden uzun olduğu için örtüşme (overlap) görünmedi. 9 parçanın 3'ü yalnızca başlıktan oluşuyor ve cevap içermiyor.

- **Embedding modeli:** Hesapta `gemini-embedding-001`, `gemini-embedding-2` ve bir önizleme sürümü listelendi, ilk ikisi çalıştı (3072 boyutlu vektör). Metin için `gemini-embedding-001` seçildi, çünkü `task_type` destekliyor ve listedeki her metin için ayrı vektör üretiyor. `gemini-embedding-2` çok kipli, ancak LangChain'in toplu belge gömme davranışıyla uyumunu doğrulamadık. İki modelin vektör uzayları uyumsuz, belgeler ve sorular aynı modelle gömülmeli.

- **Vektör veritabanı:** Parçalar Chroma'ya kaydedildi (`chroma_db` klasörü, `.gitignore` içinde). `similarity_search_with_score` bir mesafe döndürür, küçük değer daha yakın demek. Üç test sorusunun ilk sonucu doğru parçaydı. "Monitör bütçesi" sorusunda ikinci sonuç alakasızdı (mesafe 0.573'ten 0.728'e çıktı), çünkü toplam yalnızca 9 parça var.

- **RAG zinciri:** `retriever (k=3) | format_docs | prompt | gemini-3.5-flash-lite (temperature=0) | StrOutputParser`. Prompt, modele yalnızca verilen bağlamı kullanmasını ve cevap yoksa "Bu bilgi belgelerde yok." demesini söylüyor.

- **Sonuçlar:** Belgelerde cevabı olan altı sorunun hepsi doğru yanıtlandı. Cevabı belgelerde olmayan iki soru ("yemek kartı", "maaşlar ayın kaçında") için model uydurmak yerine "Bu bilgi belgelerde yok." dedi. Kıdeme bağlı izin sorusunda iki rakam birden verildi.

- **Sınırlılıklar:** Toplam 8 soru ve 9 parçalık küçük, kurgusal bir veriyle test edildi, bu sonuçlar gerçek belgelerdeki başarıyı garanti etmez. Retriever cevap olmasa da her zaman `k` parça getirir, bu yüzden kaynak listesi "cevabın dayanağı" değil "getirilen parçalar"dır. `temperature=0` çıktıyı sabitlemeye yardımcı olur ama garanti etmez. Model adları zamanla kapanabiliyor (`gemini-2.5-flash-lite` kapanmıştı).

- **RAG neden kullanılır:** Model şirketin iç yönergelerini bilmiyor. Belgeleri indeksleyip ilgili parçaları sorunun yanına koymak, modeli yeniden eğitmeden kendi belgelerimiz hakkında cevap verdirmenin pratik yolu. Bunun faydası, cevabı doğrulanabilir kaynaklara bağlamak, ama cevap kalitesi retriever'ın doğru parçayı getirmesine bağlı.